In [3]:
from collections import OrderedDict

from qiskit_metal import designs, Dict, MetalGUI
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.qubits.circle_transmon import CircTransmon

# custom components (live in the qcg-quantum-design env's qiskit_metal.qlibrary)
from qiskit_metal.qlibrary.qubits.squid_loop_jj import SquidLoopJJ
from qiskit_metal.qlibrary.qubits.squid_array import SquidArray
from qiskit_metal.qlibrary.sample_shapes.rect_cutout import RectCutout

from build_design import (
    L_TRANSMON_NH, L_PROBE_NH, CJ_TRANSMON_FF, CJ_PROBE_FF,
    QUBIT_COMPONENT_NAME, RES_LENGTH_MM,
)

design = designs.DesignPlanar({}, overwrite_enabled=True)
gui = MetalGUI(design)
design.variables['cpw_width'] = '20um'
design.variables['cpw_gap'] = '12.25um'

In [4]:
def build_squid_array(name, n_squids, pos_x_um, pos_y_um, orientation,
                      squid_gap_um, arm_width_um, josephson_gap_um,
                      segment_length_um, arm_semi_length_um, line_width_um,
                      L_j_nH, C_j_fF, jj_width_um, ground_gap_um=3.0):
    return SquidArray(design, name, options={
        'pos_x': f'{pos_x_um}um',
        'pos_y': f'{pos_y_um}um',
        'orientation': f'{orientation}',
        'n_squids': f'{n_squids}',
        'plate1_width': f'{arm_width_um}um',
        'squid_gap': f'{squid_gap_um}um',
        'segment_a_length': f'{arm_semi_length_um}um',
        'segment_a_width': f'{arm_width_um}um',
        'JJ_gap': f'{josephson_gap_um}um',
        'segment_b_length': f'{arm_semi_length_um}um',
        'segment_b_width': f'{arm_width_um}um',
        'segment_c_width': f'{arm_width_um}um',
        'segment_d_length': f'{segment_length_um}um',
        'segment_d_width': f'{line_width_um}um',
        'jj_width': f'{jj_width_um}um',
        'L_j': f'{L_j_nH}nH',
        'C_j': f'{C_j_fF}fF',
        'ground_gap': f'{ground_gap_um}um',
    })


try:
    array_1.delete()
except NameError:
    pass

array_1 = build_squid_array(
    name="array_1", n_squids=23, pos_x_um=-1290, pos_y_um=0, orientation=0.0,
    squid_gap_um=25, arm_width_um=4, josephson_gap_um=3,
    segment_length_um=6, arm_semi_length_um=15, line_width_um=9,
    L_j_nH=L_PROBE_NH, C_j_fF=CJ_PROBE_FF, jj_width_um=2.0,
)

gui.rebuild()
gui.autoscale()

In [5]:
launch_point1 = OpenToGround(design, 'launch_point1', options=dict(
    pos_x='-275um', pos_y='-850um', orientation='0',
    termination_gap='cpw_gap', gap='cpw_gap', width='cpw_width'))

launch_point2 = ShortToGround(design, 'launch_point2', options=dict(
    pos_x='-150um', pos_y='50um', orientation='0',
    termination_gap='cpw_gap', gap='cpw_gap', width='cpw_width'))

gui.rebuild()
gui.autoscale()

In [6]:
jogs = OrderedDict()
jogs[0] = ["L", "100um"]
jogs[1] = ["L", "700um"]

readout_1 = RouteMeander(design, 'readout_1', options=dict(
    pin_inputs=dict(
        start_pin=dict(component='launch_point1', pin='open'),
        end_pin=dict(component='launch_point2', pin='short'),
    ),
    fillet='40um',
    lead=dict(
        start_straight='10um',
        end_straight='1180um',
        end_jogged_extension=jogs,
    ),
    total_length=f'{RES_LENGTH_MM}mm',
))

gui.rebuild()
gui.autoscale()

/home/ruiz/miniconda3/envs/qcg-quantum-design/lib/python3.11/site-packages/qiskit_metal/qgeometries/qgeometries_handler.py:539: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.tables[kind] = pd.concat(


In [ ]:
from SQDMetal.Utilities.PVD_Shadows import PVD_Shadows
%matplotlib inline
design.chips['main']['evaporations'] = Dict(
    layer2=Dict(
        bottom_layer='200nm',
        top_layer='100nm',
        undercut='200nm',
        pvd1 = Dict(
        angle_phi = '0',
        angle_theta = '-45',
        metal_thickness = '20nm'
        ),
        pvd2 = Dict(
        angle_phi = '0',
        angle_theta = '45'
        )
    )
)
pvdSh = PVD_Shadows(design)
pvdSh.plot_layer(2,'separate', plot_mask=True)


In [ ]:
int_area = pvdSh.get_shadow_largest_interior_for_component('junction')[1]
int_area